In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score, jaccard_score, hamming_loss
from scipy.spatial.distance import hamming, cosine
import random
import time
import csv
import json


In [ ]:
arguments = pd.read_csv("arguments.tsv", sep="\t")
ground_truth = pd.read_csv("labels-level1.tsv", sep="\t")

with open('values.json', 'r') as f:
    values_dict = json.load(f)

print(values_dict)

def build_values_prompt_text(values_dict):
    lines = []

    for v in values_dict["values"]:
        name = v["name"]
        vid = v["id"]
        desc = "; ".join(v["descriptions"]) 

        line = f"{name} — {desc}"
        lines.append(line)

    return "\n".join(lines)

VALUES_LIST =build_values_prompt_text(values_dict)


{'values': [{'alternativeName': 'Creativity/Imagination', 'name': 'Be creative', 'id': 'SDT1', 'level2': 'Self-direction: thought', 'level3': ['Openness to change'], 'level4a': ['Personal focus'], 'level4b': ['Growth, Anxiety-free'], 'descriptions': ['allowing for more creativity or imagination', 'being more creative', 'fostering creativity', 'promoting imagination']}, {'alternativeName': 'Curious/Interested', 'name': 'Be curious', 'id': 'SDT2', 'level2': 'Self-direction: thought', 'level3': ['Openness to change'], 'level4a': ['Personal focus'], 'level4b': ['Growth, Anxiety-free'], 'descriptions': ['being the more interesting option', 'fostering curiosity', 'making people more keen to learn', 'promoting discoveries', 'sparking interest']}, {'alternativeName': 'Freedom of thought', 'name': 'Have freedom of thought', 'id': 'SDT3', 'level2': 'Self-direction: thought', 'level3': ['Openness to change'], 'level4a': ['Personal focus'], 'level4b': ['Growth, Anxiety-free'], 'descriptions': ['al

In [13]:
print(VALUES_LIST)

Be creative — allowing for more creativity or imagination; being more creative; fostering creativity; promoting imagination
Be curious — being the more interesting option; fostering curiosity; making people more keen to learn; promoting discoveries; sparking interest
Have freedom of thought — allowing people to figure things out on their own; allowing people to make up their mind; resulting in less censorship; resulting in less influence on people's thoughts
Be choosing own goals — allowing people to choose what is best for them; allowing people to decide on their life; allowing people to follow their dreams
Be independent — allowing people to plan on their own; resulting in fewer times people have to ask for consent
Have freedom of action — allowing people to be self-determined; allowing people to do things even though this may hurt them in the long run; resulting in more times people can do what they want
Have privacy — allowing for private spaces; allowing for time alone; resulting 

### Evaluate 503 arguments, USA test

In [ ]:
test_arguments = arguments[
    (arguments["Part"] == "usa") &
    (arguments["Usage"] == "test")
]["Argument ID"].tolist()

print(f"There are {len(test_arguments)} arguments in total in the test set of USA")

There are 503 arguments in total in the test set of USA


In [15]:
arguments_503 = arguments[arguments["Argument ID"].isin(test_arguments)]
arguments_503.to_csv("arguments_503.csv", index=False)

ground_truth_503 = ground_truth[ground_truth["Argument ID"].isin(test_arguments)]
ground_truth_503.to_csv("labels-level1_503.csv", index=False)

In [16]:
values = list(ground_truth.columns[1:]) 
print(f"Level 1 has {len(values)} values in total.")

Level 1 has 54 values in total.


In [ ]:
from ollama import Client

def create_client():
    url_server = "" #your ollama server url
    client = Client(host=url_server, verify=False)
    return client

def ask_model(client, model, prompt):
    
    messages = [{"role": "user", "content": prompt}]
    response = client.chat(model=model, messages=messages)
    content = response['message']['content']

    return content.strip().upper()

In [18]:
FULL_PROMPT = """
You are given an argument:

Conclusion: {CONCLUSION}
Stance: {STANCE}
Premise: {PREMISE}

You are also given a list of 54 human values with their descriptions.

VALUES (in fixed order):
{VALUES_LIST}

Task:
For each value in the list, decide whether it could be used as a justification for the argument.

Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
If asked 'Why is this good?', might this be their justification? "Because it is good to "VALUE"".

Rules:
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Make sure you understand the examples given

Examples:
Example arguments against "Social media should be banned".
- Argument: We have to be honest. Social media does not make people polite. But it makes our lives easier and more interesting.	
- Select all justifications one could provide: (✔ have a comfortable life) (from "easier lives"), (✔ have pleasure) (also from "easier lives"), (✔ have an exciting life) (from "more interesting"), (✔ have a varied life) (also from "more interesting"). But do not select justifications for concessions ((✖ be polite)) or empty phrases ((✖ be honest), (✖ be logical), (✖ have an objective view) for "We have to be honest").
Example arguments in favor of "Social media should be banned".
- Argument: Through social media people can spread biased opinions on topics or misinform the general public.	
- Use the examples for each justification to get a better understanding of the justifications ((✔ have freedom of thought) from reduced misleading influence on people's thoughts). But do not select justifications only because they are connected to the topic in general ((✖ have privacy) for the general threat of social media to privacy: it is not mentioned here).

Return format:
- Return a list of EXACTLY 54 integers, each integer must be 0 or 1:
    - 1 = YES (this value is relevant)
    - 0 = NO (this value is not relevant)
- The order MUST match exactly the order of the VALUES list: ["BE CREATIVE", "BE CURIOUS", "HAVE FREEDOM OF THOUGHT", "BE CHOOSING OWN GOALS", "BE INDEPENDENT", "HAVE FREEDOM OF ACTION", "HAVE PRIVACY", "HAVE AN EXCITING LIFE", "HAVE A VARIED LIFE", "BE DARING", "HAVE PLEASURE", "BE AMBITIOUS", "HAVE SUCCESS", "BE CAPABLE", "BE INTELLECTUAL", "BE COURAGEOUS", "HAVE INFLUENCE", "HAVE THE RIGHT TO COMMAND", "HAVE WEALTH", "HAVE SOCIAL RECOGNITION", "HAVE A GOOD REPUTATION", "HAVE A SENSE OF BELONGING", "HAVE GOOD HEALTH", "HAVE NO DEBTS", "BE NEAT AND TIDY", "HAVE A COMFORTABLE LIFE", "HAVE A SAFE COUNTRY", "HAVE A STABLE SOCIETY", "BE RESPECTING TRADITIONS", "BE HOLDING RELIGIOUS FAITH", "BE COMPLIANT", "BE SELF-DISCIPLINED", "BE BEHAVING PROPERLY", "BE POLITE", "BE HONORING ELDERS", "BE HUMBLE", "HAVE LIFE ACCEPTED AS IS", "BE HELPFUL", "BE HONEST", "BE FORGIVING", "HAVE THE OWN FAMILY SECURED", "BE LOVING", "BE RESPONSIBLE", "HAVE LOYALTY TOWARDS FRIENDS", "HAVE EQUALITY", "BE JUST", "HAVE A WORLD AT PEACE", "BE PROTECTING THE ENVIRONMENT", "HAVE HARMONY WITH NATURE", "HAVE A WORLD OF BEAUTY", "BE BROADMINDED", "HAVE THE WISDOM TO ACCEPT OTHERS", "BE LOGICAL", "HAVE AN OBJECTIVE VIEW"]
- Do NOT skip any value and do NOT add text, explanation, or formatting

Example output:
[0, 1, 0, 0, 1, 0, ..., 0] (with the length of the list being EXACTLY 54 and the order matching the VALUES list)
"""


TEST_PROMPT = """
You are given an argument:

Conclusion: {CONCLUSION}
Stance: {STANCE}
Premise: {PREMISE}

You are also given a list of 54 human values with their descriptions.

VALUES (in fixed order):
{VALUES_LIST}

Task:
For each value in the list, decide whether it could be used as a justification for the argument.

Imagine someone is arguing "{STANCE}" "{CONCLUSION}" by saying: "{PREMISE}"
If asked 'Why is this good?', might this be their justification? "Because it is good to "VALUE"".

Rules:
- Typically, one could provide at least 1 and not more than 5 of these justifications for an argument. If you would select more than 10 justifications for an argument, reduce your selection to the most fitting ones.
- Make sure you understand the examples given

Examples:
Example arguments against "Social media should be banned".
- Argument: We have to be honest. Social media does not make people polite. But it makes our lives easier and more interesting.	
- Select all justifications one could provide: (✔ have a comfortable life) (from "easier lives"), (✔ have pleasure) (also from "easier lives"), (✔ have an exciting life) (from "more interesting"), (✔ have a varied life) (also from "more interesting"). But do not select justifications for concessions ((✖ be polite)) or empty phrases ((✖ be honest), (✖ be logical), (✖ have an objective view) for "We have to be honest").
Example arguments in favor of "Social media should be banned".
- Argument: Through social media people can spread biased opinions on topics or misinform the general public.	
- Use the examples for each justification to get a better understanding of the justifications ((✔ have freedom of thought) from reduced misleading influence on people's thoughts). But do not select justifications only because they are connected to the topic in general ((✖ have privacy) for the general threat of social media to privacy: it is not mentioned here).

Return format:
- Return a list of the exactly 54 values that I have passed to you, I only want the exact names
- The order MUST match exactly the order of the VALUES list
- Do NOT skip any value and do NOT add text, explanation, or formatting

Example output:
["value1", "value2", "value3", ..., "value54"] with "valueX" being either the name of the value
"""

In [ ]:
import re

def parse_llm_output(output, expected_len=54):
    try:
        nums = re.findall(r'\d+', output)
        nums = [int(n) for n in nums]

        if len(nums) != expected_len:
            raise ValueError(f"Expected {expected_len} values, got {len(nums)}")

        return nums

    except Exception as e:
        print("Parsing error:", e)
        return None
    
def safe_ask_model(client, model, prompt, retries=3):
    for _ in range(retries):
        output = ask_model(client, model, prompt)
        parsed = parse_llm_output(output)
        if parsed is not None:
            return parsed
    return None


In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import time

def run_experiment(client, model, values_dict, arguments, output_path="results.csv"):

    value_names = [v["name"] for v in values_dict["values"]]

    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        done_ids = set(existing_df["Argument ID"].tolist())
        results = existing_df.values.tolist()
        print(f"Resuming... {len(done_ids)} arguments already processed.")
    else:
        done_ids = set()
        results = []

    segons_llista = []

    for _, row in tqdm(arguments.iterrows(), total=len(arguments)):

        arg_id = row["Argument ID"]

        if arg_id in done_ids:
            continue

        start = time.time()

        prompt = FULL_PROMPT.format(
            CONCLUSION=row["Conclusion"],
            STANCE=row["Stance"],
            PREMISE=row["Premise"],
            VALUES_LIST=VALUES_LIST
        )

        vector = safe_ask_model(client, model, prompt)

        end = time.time()
        segons = end - start

        segons_llista.append(segons)

        results.append([arg_id] + vector + [segons])

        df = pd.DataFrame(
            results,
            columns=["Argument ID"] + value_names + ["seconds"]
        )

        df.to_csv(output_path, index=False)

    return pd.DataFrame(results, columns=["Argument ID"] + value_names + ["seconds"])

In [ ]:
model = "gpt-oss:20b"

client = create_client()
arguments = pd.read_csv("arguments_503.csv")
output = "results_gptoss.csv"
results = run_experiment(client, model, values_dict, arguments, output)